# 1 — Extraction and Ingestion

We take one PDF and turn it into something a search engine can use.

```
 PDF
  |
  v
parse       docling_io.parse_pdf        six models -> a typed object graph
  |
  v
inspect     inspection.inspect          did it work? counts and warnings
  |         inspection.write_extraction_report
  v
figures     docling_io.save_figures     store the rendered chart images
  |
  v
chunk       chunking.build_records      split, add context, summarise tables
  |
  v
index       sync_module.sync            embed what changed, upsert, delete
```

**Nothing is embedded until the last step.** Parsing, inspecting and chunking
all happen before a single vector is computed — which is the point of the
checkpoints: a bad parse is caught while it is still free to fix.

Each step has a checkpoint. That matters more than it sounds, and here is why.

**When this pipeline goes wrong, it usually does not crash.** It keeps running and
produces an index that looks complete but is missing your tables, or your charts, or
your equations. You only find out weeks later when the answers are wrong.

So after every step we stop and check.

This notebook uses the `rag` package. The same package runs on AWS, so what you test
here is exactly what runs in production. There is no second copy that can drift.

    rag.config       settings that both halves of the pipeline must agree on
    rag.docling_io   reading the PDF
    rag.inspect      checking the parse worked, and writing reports
    rag.tables       tables: reading them, checking them, summarising them
    rag.chunking     splitting into pieces, with metadata
    rag.index        talking to the vector database
    rag.sync         working out what changed since last time

| § | Step | What to check |
|---|---|---|
| 0 | Setup | your settings |
| 1 | Parse | which models ran |
| 2 | Inspect | every figure described, no broken tables |
| 3 | Read the report | against the actual PDF |
| 4 | Figures | images saved |
| 5 | Chunk | table summaries made, nothing cut off |
| 6 | Read the chunks | what will actually be searched |
| 7 | Index | added / unchanged / removed |
| 8 | Verify | settings recorded |

---
## 0. Setup

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os

os.environ.setdefault("OPENAI_API_KEY", "")
os.environ.setdefault("PINECONE_API_KEY", "")

# Read once at import time, so set these before importing the rag package.
os.environ.setdefault("CHUNK_TOKEN_TARGET", "1024")     # chunk size, in tokens
os.environ.setdefault("FIGURE_AREA_THRESHOLD", "0.01")  # min page fraction for a figure
os.environ.setdefault("REPORT_DIR", "reports")

In [ ]:
import json
from pathlib import Path

import pandas as pd

from rag import chunking, clients, config, docling_io, tables
from rag import index as index_module
from rag import inspect as inspection
from rag import sync as sync_module

SOURCE_PDF = Path("pdfs/AI-Enablers-Adopters-research-report.pdf")

# Set to your bucket to store figure images and share the parse cache.
# Leave as None to keep everything local.
BUCKET = None

DOC_ID = config.slugify(SOURCE_PDF.stem)

print(f"document   : {DOC_ID}")
print(f"embedding  : {config.EMBED_MODEL} ({clients.EMBED_DIMS}d)")
print(f"chunk size : {config.CHUNK_TOKENS} tokens")
print(f"vision     : {config.VISION_MODEL}")
print(f"reports    : {config.REPORT_DIR.resolve()}")

---
## 1. Parse

### What "parsing" actually means here

It is not one thing. It is six models running one after another.

| Model | What it does |
|---|---|
| Layout | Finds the boxes on each page and labels them: heading, paragraph, table, picture |
| TableFormer | Works out the rows and columns inside a table |
| OCR | Reads text from scanned pages that have no text layer |
| Figure classifier | Says whether a picture is a chart, a photo or a logo |
| CodeFormula | Reads equations and code |
| Vision model | Writes a description of each chart |

Only the last one costs money. The other five download once and run locally.

### The trap

**Almost all of these are switched off by default.**

If you use Docling's default settings, you get text and not much else. An equation
becomes the placeholder `formula-not-decoded` and its content is simply gone. No
error. Nothing in the log. You would never know.

That is why the next cell prints every available setting before we choose any.

### One thing to know before you run it

**This cell re-reads the PDF every time.** That means one vision call per chart —
seconds for a small document, minutes for a big one.

Run it once. Keep `doc` in memory. Everything below works on that and is instant.
Only come back here if you change a parse setting.

There is no cache, on purpose. A cache would be keyed on the filename — but the
result depends on your settings too. Change a setting and a filename cache hands
back the old result, so it looks like your change did nothing.

In [ ]:
from docling.datamodel.pipeline_options import PdfPipelineOptions

# What this Docling build supports, and what it defaults to. Flag names move
# between releases, which is why rag.docling_io reads them rather than assuming.
for name, field in sorted(PdfPipelineOptions.model_fields.items()):
    if name.startswith(("do_", "generate_", "enable_")):
        print(f"  {name:36s} {field.default}")

In [ ]:
# Six models run here, one after another. On a short document this is seconds;
# on a 250-page protocol with many figures it is minutes, because the vision
# model is called once per figure.
#
# This cell does NOT chunk and does NOT embed. It produces `doc` — an object
# graph of typed elements — and everything below works on that.
#
# Run it once. Re-running costs the vision calls again.
doc = docling_io.parse_pdf(SOURCE_PDF)

In [ ]:
# Rendering the whole document to markdown is expensive on long PDFs, so do it once
# here and reuse it for the checks below and for the date heuristic.
markdown = doc.export_to_markdown()
doc_date = chunking.document_date(SOURCE_PDF, markdown[:4000])

print(f"pages {len(doc.pages)}   document date {doc_date}")

### What one element of each type looks like

Docling does not hand you a big string of text. It hands you a set of **objects**,
and each one knows what it is.

That is the whole reason this works. A table object can be asked for its cells. A
picture object can be asked for its description. You never have to guess from the
text what something was.

The next cell reads the fields straight off the objects, rather than showing a list
I picked. So what you see is really there — including fields we never use.

Three things to look for:

**`self_ref`** is the object's ID. It is how a table gets linked to its summary later,
and how a figure gets linked to its saved image.

**`prov` is a list, not one value.** One entry per page the element appears on. That
is why a chunk can have both a `page` and a `page_end`.

**`data.table_cells`** is the real table, before it gets written out as text. Each
cell knows if it spans columns and if it is a header. `export_to_markdown()` throws
some of that away — which is exactly the loss `table_looks_broken()` is watching for.

And one thing that will bite you later:

**The object's Python type is not always what its label says.** `doc.iterate_items()`
gives you real `TableItem` objects. But `chunk.meta.doc_items` — which you meet in the
chunking section — gives you lighter stand-ins that say `label=TABLE` but fail an
`isinstance` check. Check the type there instead of the label and you get nothing
back, silently, for every chunk.

In [ ]:
# Every field on one sample of every element type the layout model found.
#
# Nothing here is hand-picked. The fields are read off the objects themselves, so
# what you see is the actual shape of the graph everything downstream works on —
# including fields this pipeline never touches, which is worth knowing when you go
# looking for something it does not currently use.

from docling_core.types.doc import PictureItem, TableItem


def preview(value, width: int = 88) -> str:
    """One line describing a field's value: its type, size, and a readable sample."""
    if value is None:
        return "None"
    if isinstance(value, (str, int, float, bool)):
        text = str(value).replace("\n", " ")
        return f"{text[:width]}{'…' if len(text) > width else ''}"
    if isinstance(value, (list, tuple)):
        if not value:
            return f"{type(value).__name__}[0] (empty)"
        inner = type(value[0]).__name__
        return f"{type(value).__name__}[{len(value)}] of {inner}"
    if isinstance(value, dict):
        return f"dict[{len(value)}] keys={list(value)[:6]}"
    return type(value).__name__


def dump_fields(obj, indent: str = "  ") -> None:
    """Print every declared field of a pydantic object, with its type and value.

    Docling's items are pydantic models, so `model_fields` is the authoritative
    list — better than dir(), which mixes in methods and validators. The fallback
    covers anything that is not a model.
    """
    fields = getattr(type(obj), "model_fields", None)
    if fields:
        for name in fields:
            value = getattr(obj, name, None)
            annotation = fields[name].annotation
            type_name = getattr(annotation, "__name__", str(annotation))
            print(f"{indent}{name:<18} {str(type_name)[:34]:<36} {preview(value)}")
    else:
        for name, value in vars(obj).items():
            if not name.startswith("_"):
                print(f"{indent}{name:<18} {type(value).__name__:<36} {preview(value)}")


seen = {}
for item, level in doc.iterate_items():
    label = str(getattr(item, "label", "?")).rsplit(".", 1)[-1]
    seen.setdefault(label, (item, level))

print(f"{len(seen)} element types found: {', '.join(sorted(seen))}\n")

for label in sorted(seen):
    item, level = seen[label]

    print("=" * 100)
    print(f"{label}   ({type(item).__name__}, tree level {level})")
    print("=" * 100)

    print("\nFIELDS")
    print(f"  {'name':<18} {'declared type':<36} value")
    print(f"  {'-'*18} {'-'*36} {'-'*40}")
    dump_fields(item)

    # prov is a list of provenance records, one per page the element appears on.
    # Its fields are where page numbers and bounding boxes actually live.
    prov = (getattr(item, "prov", None) or [None])[0]
    if prov is not None:
        print("\nprov[0]")
        dump_fields(prov, indent="  ")
        bbox = getattr(prov, "bbox", None)
        if bbox is not None:
            print("\n  prov[0].bbox")
            dump_fields(bbox, indent="    ")

    # Type-specific structure that the field list only hints at.
    if isinstance(item, TableItem):
        data = getattr(item, "data", None)
        if data is not None:
            print("\ndata  (the grid itself)")
            dump_fields(data, indent="  ")
            cells = getattr(data, "table_cells", None) or []
            if cells:
                print("\n  data.table_cells[0]")
                dump_fields(cells[0], indent="    ")
        markdown = item.export_to_markdown(doc)
        print(f"\nexport_to_markdown()  —  {len(tables.table_cells(markdown))} cells, "
              f"structure {tables.table_looks_broken(markdown) or 'looks sound'}")
        for line in markdown.splitlines()[:4]:
            print(f"  {line[:96]}")

    elif isinstance(item, PictureItem):
        annotations = docling_io.picture_annotations(item)
        print(f"\nannotations  —  {len(annotations)}")
        for n, annotation in enumerate(annotations):
            print(f"\n  annotations[{n}]  ({type(annotation).__name__})")
            dump_fields(annotation, indent="    ")
        print(f"\nget_image(doc) : "
              f"{'a PIL image' if item.get_image(doc) is not None else 'None — not rendered'}")

    print()

---
## 2. Inspect the parse

This is the checkpoint. It matters because **extraction failures do not raise an
error.**

If a setting was off, you get an empty description or a placeholder. Every step after
that runs perfectly happily on top of it. You end up with an index that looks
finished and is quietly missing content. Nothing downstream can spot it.

So we stop here and count things.

In [ ]:
report = inspection.inspect(doc, markdown)

Four numbers to read.

**`pictures_described` should equal `pictures`.** If it is 0, chart descriptions did
not run and every chart in your document is invisible to search. Usual causes:
`enable_remote_services` is off, `generate_picture_images` is off, or
`picture_area_threshold` is filtering everything out.

**`formulas_undecoded` should be 0** if your document has equations in it.

**`tables_suspect` should be 0.** Anything above zero means a table came out with a
broken grid — row labels merged into number columns, or a header row duplicated.

That is worse than a table that failed completely. A broken grid still gets indexed,
and its summary describes a table that does not exist. Those tables are re-read from
a picture instead, which usually recovers them.

**`charts_with_data`** counts figures where the numbers were pulled out, not just
described. Zero is common — chart extraction only works on charts saved as images,
and many financial and research PDFs draw them as line art instead. If it is zero
across your whole corpus, that model is costing you time for nothing.

---
## 3. Read the extraction

`pages 7` tells you the file opened. It tells you nothing about whether the content
is any good.

So we write a report you can actually read. It lists every element in order, with its
page number, its table text, and its chart description. It is written **before any
chunking and before you spend anything on embeddings**.

Anything that failed gets a banner where it should have been, and all the failures are
listed again at the top of the file.

That last part is deliberate. A missing chart description is an *absence* — and an
absence is exactly the thing your eye slides past. So we make it visible.

Open `reports/<doc>.extract.md` next to the PDF and read them side by side.

In [ ]:
report_path = inspection.write_extraction_report(doc, DOC_ID, SOURCE_PDF)

# The problems block sits at the top of the file, so this is the whole triage.
print(report_path.read_text()[:2000])

In [ ]:
# The same inventory as data, for checking a whole corpus at once rather than
# reading twenty reports.
inventory = json.loads(report_path.with_suffix(".json").read_text())

figures = [e for e in inventory["elements"] if e["kind"] == "FIGURE"]
tables = [e for e in inventory["elements"] if e["kind"] == "TABLE"]

print(f"{inventory['doc_id']}: {inventory['pages']} pages")
print(f"elements: {inventory['counts']}\n")
print(f"figures described : {sum(1 for f in figures if f.get('description'))}/{len(figures)}")
print(f"charts with data  : {sum(1 for f in figures if f.get('has_chart_data'))}/{len(figures)}")
print(f"tables serialised : {sum(1 for t in tables if t.get('cells', 0) > 0)}/{len(tables)}")
print(f"tables suspect    : {sum(1 for t in tables if t.get('structure_problems'))}/{len(tables)}")

if inventory["problems"]:
    print("\nproblems:")
    for problem in inventory["problems"]:
        print(f"  {problem}")
else:
    print("\nno extraction problems detected")

In [ ]:
# Read one figure description against the actual chart. This is the only check on
# whether the vision model read the numbers or wrote plausible prose about them.
for figure in figures:
    if figure.get("description"):
        print(f"FIGURE p{figure['page']}\n{figure['description']}\n")
        break

# And one table, with any structure warning attached.
for table in tables:
    if table.get("markdown"):
        print(f"TABLE p{table['page']}  {table['cells']} cells")
        if table.get("structure_problems"):
            print(f"  STRUCTURE SUSPECT: {'; '.join(table['structure_problems'])}")
        print(table["markdown"][:900])
        break

---
## 4. Figure images

The chart images have already been created — the vision model needed them to write
its descriptions.

If we throw them away, your app can quote a description of a chart but never show the
chart. So we save them.

Skipped if `BUCKET` is `None`.

In [ ]:
# The images already exist — the vision model needed them to write its
# descriptions. This saves them so an answer can SHOW a chart rather than only
# quote a description of one.
#
# Returns {element_ref: s3_uri}. Empty when BUCKET is None, so the cell below
# works either way.
figure_uris = docling_io.save_figures(doc, DOC_ID, BUCKET)
print(f"{len(figure_uris)} figure images stored")

---
## 5. Chunk

A chunk is a piece of the document small enough to search. How you cut it decides
what you can find.

`HybridChunker` cuts on the document's own structure first — headings, paragraphs,
tables. Then it checks each piece against a token limit and splits anything too big.

Then `contextualize()` adds the heading path to the front of each chunk. **That
combined string is what gets embedded.** So the section a chunk came from becomes
part of what you can search for.

### Why every table gets an extra chunk

Here is the problem. Take this table:

    | Q1    | Q2    | Q3    | Q4    |
    | $100M | $110M | $125M | $140M |

Now ask: *"how did revenue grow through the year?"*

The words **revenue**, **grow** and **year** do not appear anywhere in that table.
Not once. So a search for that question will never find it. The table is sitting in
your index, completely unreachable.

The fix: for every table, we also write a sentence describing it. *"Quarterly revenue,
rising from $100M in Q1 to $140M in Q4, a 40% increase."* Now the words match.

The raw rows stay too, as separate chunks, linked by `table_id`. So the search finds
the description, and the exact numbers are one step away.

The summary can also say things no single cell contains. A schedule listing weeks 0,
4, 8 and 12 never contains the words "every 4 weeks" — but that is what somebody will
search for.

**And if the table's grid came out broken,** the summary is written by looking at a
picture of the table instead. Describing a broken grid confidently is worse than not
describing it at all.

In [ ]:
# doc -> records. Still no embeddings.
#
# Each record is {"text": ..., "meta": {...}}: the exact string that will be
# embedded, plus everything you might later want to filter on.
#
# Watch the printed line. `table_summary` should equal your table count — one
# summary per table, regardless of how many fragments the chunker made from it.
records = chunking.build_records(doc, SOURCE_PDF, DOC_ID, doc_date, figure_uris)

In [ ]:
# The chunk mix, as a table. Three things to read here.
#
# `table_summary` count  -> should match the number of tables in the document
# `from image`           -> summaries written by LOOKING at a rendered table,
#                           because its parsed grid was unusable
# `over budget`          -> should be 0. Anything here was truncated, and the
#                           tail of that chunk is gone.
frame = pd.DataFrame([{
    "pos":      r["meta"]["position"],
    "type":     r["meta"]["content_type"],
    "tokens":   r["meta"]["n_tokens"],
    "page":     r["meta"]["page"],
    "table_id": r["meta"]["table_id"] or "",
    "source":   r["meta"].get("summary_source", ""),
} for r in records])

print(frame.groupby("type")["tokens"].agg(["count", "mean", "max"]).round(0).to_string())

n_summaries = int((frame["type"] == "table_summary").sum())
n_tables = frame.loc[frame.type == "table", "table_id"].nunique()
print(f"\ntable summaries : {n_summaries} for {n_tables} tables")
print(f"from image      : {int((frame['source'] == 'image').sum())}")
print(f"truncated       : {sum(1 for r in records if r['meta'].get('truncated'))}")
print(f"over budget     : {int((frame.tokens > config.CHUNK_TOKENS).sum())}")

In [ ]:
# A summary and its fragments, in reading order. The summary should sit immediately
# before the rows it describes, and share their table_id.
summaries = [r for r in records if r["meta"]["content_type"] == "table_summary"]

if summaries:
    table_id = summaries[0]["meta"]["table_id"]
    for record in records:
        if record["meta"]["table_id"] == table_id:
            meta = record["meta"]
            source = f"  [{meta['summary_source']}]" if meta.get("summary_source") else ""
            print(f"[{meta['position']:>3}] {meta['content_type']:<14} p{meta['page']}{source}")
            print(f"      {record['text'][:200].replace(chr(10), ' ')}\n")
else:
    print("no table summaries in this document")

---
## 6. Read the chunks

The last report answered *did the parse work*.

This one answers a different question: **is the text we are about to embed the right
text?** It shows every chunk exactly as it will be stored, with its token count, and
each table summary sitting right before the rows it describes.

In [ ]:
chunk_report = inspection.write_chunk_report(records, DOC_ID)
print(chunk_report.read_text()[:1800])

In [ ]:
# One chunk of each type, read end to end.
for wanted in ("text", "table_summary", "table", "figure", "formula", "code"):
    record = next((r for r in records if r["meta"]["content_type"] == wanted), None)
    if record is None:
        continue
    meta = record["meta"]
    print("=" * 78)
    print(f"{meta['chunk_id']}  {wanted}  p{meta['page']}-{meta['page_end']}  "
          f"{meta['n_tokens']} tok")
    print(f"headings: {meta['headings']}")
    if meta.get("image_uri"):
        print(f"image: {meta['image_uri']}")
    print("-" * 78)
    print(record["text"][:600], "\n")

---
## 7. Index

Each chunk gets an ID built from its own text:

    {doc_id}:{sha256(text)[:16]}:{occurrence}

That looks like a small detail. It is not — it changes what re-ingesting costs.

**If IDs were positions** (chunk 1, chunk 2, chunk 3), then inserting one paragraph
on page 2 shifts every ID after it. Every chunk looks new. You re-embed the whole
document to fix one paragraph.

**Because IDs come from the text**, only the chunks whose text actually changed get
new IDs. Everything else keeps its old ID and its old vector. Re-ingesting becomes a
comparison: what is new, what is gone, what is unchanged.

The other two parts:

**`doc_id` at the front** stops two documents colliding. If both contain the same
legal disclaimer, that paragraph would otherwise get the same ID in both — and the
second upload would silently overwrite the first document's chunk.

**`occurrence` at the end** handles text that repeats inside one document, like a
footer on every page. Each copy stays separately findable with its own page number.

One useful side effect: running this twice is safe. The second run writes identical
vectors and deletes nothing.

In [ ]:
# THIS is where embeddings happen and money is spent — not in any cell above.
#
# open_index() probes the embedding dimension at connect time. Querying an index
# built at a different dimension returns plausible scores with no error, which
# is the hardest failure in this pipeline to notice.
#
# sync() then compares what is in the index against what we just built:
#
#     unchanged  do nothing
#     added      embed and upsert
#     removed    delete
#     moved      rewrite metadata, no embedding call
index = index_module.open_index(create=True)
plan = sync_module.sync(index, DOC_ID, records)

In [ ]:
# Run this cell again: everything should be unchanged and nothing re-embedded.
sync_module.sync(index, DOC_ID, records)

---
## 8. Verify

The manifest records how this index was built — which embedding model, which chunk
size, which settings.

The retrieval notebook reads it and refuses to start if its own settings disagree.

That check is worth having because the failure it prevents is invisible. Search an
index with the wrong embedding model and you get results back, with scores that look
perfectly reasonable, that happen to be meaningless.

In [ ]:
# How this index was built: embedding model, chunk size, vision model.
#
# The retrieval notebook reads this and REFUSES to start if its own settings
# disagree. Not a warning — the failure it prevents has no visible symptom.
manifest = index_module.write_manifest(
    DOC_ID, len(records), extra={"source": SOURCE_PDF.name, "doc_date": doc_date})
print(json.dumps(manifest, indent=2))

Done.

To add another document, change `SOURCE_PDF` and run from §1 again. Each document
gets its own space in the index and its own entry in the manifest.

Now go to `02_retrieval.ipynb`.